## Импортируем необходимые библиотеки

In [1]:
import re
from datetime import datetime

import pandas as pd 
import numpy as np
import geopandas as gpd

import seaborn as sns
import matplotlib.pyplot as plt

import phik
from phik.report import plot_correlation_matrix

pd.set_option('display.max_columns', None)

Прочитаем файл с данными

In [2]:
df = pd.read_csv('./data/listings_preprocessed.csv', low_memory=False)
df.shape

(142285, 52)

Привяжем объявления к административным районам Москвы. Загрузим GeoJSON-файл с границами районов Москвы и установим систему координат. Затем на основе широты и долготы из датафрейма мы создали GeoDataFrame с точечной геометрией и с помощью пространственного соединения сопоставили каждой точке название района, внутри которого она находится. А также очистим полученные названия районов от лишних слов ("р-н", "район") и посмотрим долю объявлений с заполненным районом.


In [3]:
raions = gpd.read_file('./data/moscow_raions.geojson')[['name', 'geometry']].set_crs(4326)
pts = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df['lon'], df['lat']), crs=4326)
joined = gpd.sjoin(pts, raions, how='left', predicate='within')
geo_district = joined[~joined.index.duplicated(keep='first')].reindex(df.index)['name']
geo_district = geo_district.str.replace(r'^(р-н|район)\s+', '', regex=True).str.replace(r'\s+район$', '', regex=True).str.strip()
df['district'] = df['district'].where(df['district'].notna(), geo_district.values)
df['district'].notna().mean()

np.float64(0.4603436764240784)

In [4]:
df.loc[(df['ceiling_height'] < 2) | (df['ceiling_height'] > 8), 'ceiling_height'] = np.nan
bad_geometry = df['living_area'].fillna(0) + df['kitchen_area'].fillna(0) > df['total_area']
df.loc[bad_geometry, ['living_area', 'kitchen_area']] = np.nan
df = df[df['total_area'] >= 10]
df.shape

(142285, 52)

Мы заменили на пропуски аномальные значения высоты потолков (менее 2 м и более 8 м), а также обнулили жилую площадь и площадь кухни в случаях, когда их сумма превышала общую площадь. Дополнительно удалили объекты с общей площадью менее 10 кв.м.

### Заполнение пропусков

In [5]:
text_cols = [c for c in df.columns if df[c].dtype == 'str']
for col in text_cols:
    mask_1 = df[col].str.strip().str.lower().isin(['', 'nan', 'none'])
    mask_2 = df[col].isna()
    df.loc[mask_1 | mask_2, col] = np.nan 

for c in ['municipality', 'region', 'district', 'parking', 'window_view', 'renovation', 'building_type', 'flat_type']:
    df[c] = df[c].fillna('unknown')

for c in ['deal_conditions', 'seller_type']:
    df[c] = df[c].fillna(df[c].mode()[0])

b_map = {'True': 1, 'False': 0, True: 1, False: 0, 't': 1, 'f': 0}
for c in ['is_apartments', 'is_new_building', 'phone_protected', 'is_studio', 'mortgage_allowed', 'nearest_metro_walk']:
    df[c] = df[c].map(b_map).fillna(0).astype('int8')

df.shape

(142285, 52)

Заполнили пропуски в текстовых столбцах, а также проставили unknown для данных, для которых не можем получить достоверную "альтернативу", заполнили модой необходимые поля и привели булевы строки к 0/1.

In [6]:
df.loc[df['is_studio'] == 1, 'rooms'] = 0
df['rooms'] = df['rooms'].fillna(df['rooms'].median())
df.shape

(142285, 52)

In [7]:
group_cols = ['building_type', 'year_built', 'total_floors']

ch_group = df.groupby(group_cols)['ceiling_height'].transform('median')
ch_glob = df['ceiling_height'].median()

df['ceiling_height'] = df['ceiling_height'].fillna(ch_group).fillna(ch_glob)

Мы сгруппировали объявления по типу здания, году постройки и количеству этажей, и для каждой группы вычислили медианное значение высоты потолков. Затем заполнили пропуски в высоте потолков сначала групповой медианой, а затем глобальной медианой по всему датафрейму.

In [8]:
df['area_bin'] = (df['total_area'] // 5) * 5

for col in ['kitchen_area', 'living_area']:
    med_by = df.groupby(['building_type', 'area_bin'])[col].transform('median')
    df[col] = df[col].fillna(med_by).fillna(df[col].median())

df = df.drop(columns='area_bin')

df.loc[df['year_built'] < 1500, 'year_built'] = np.nan
yb_btype = df.groupby('building_type')['year_built'].transform('median')
yb_dist = df.groupby(df['district'].where(df['district'] != 'unknown'))['year_built'].transform('median')
yb_muni = df.groupby(df['municipality'].where(df['municipality'] != 'unknown'))['year_built'].transform('median')
yb_global = df['year_built'].median()

df['year_built'] = df['year_built'].fillna(yb_btype).fillna(yb_dist).fillna(yb_muni).fillna(yb_global).astype('int32')

Для кухонной и жилой площади мы заполнили пропуски медианой по группам "тип здания + группа по общей площади", а оставшиеся — глобальной медианой. Для года постройки мы очистили значения младше 1500 года и заполнили пропуски последовательно: медианой по типу здания, району, муниципалитету и, наконец, общей медианой.

### Создаём признаки

In [9]:
df['building_age'] = datetime.now().year - df['year_built']
df['completion_date'] = df['completion_date'].str.extract(r'(\d{4})').astype(float)

current_year = datetime.now().year
df['is_ready'] = ~((df['is_new_building'] == 1) & (df['completion_date'] > current_year))

df['completion_year'] = df['completion_date']
df['years_to_completion'] = df['completion_date'] - current_year
df['is_presale'] = (df['completion_date'] > current_year).astype('int8')
df['has_completion'] = df['completion_date'].notna().astype('int8')

df['had_discount'] = (df['price_last'] < df['price_first']).astype('int8')
df['price_drop_pct'] = (df['price_first'] - df['price_last']) / df['price_first']


Мы рассчитали возраст здания, извлекли год завершения строительства и создали бинарные признаки: готово ли здание, продаётся ли на этапе строительства, известна ли дата завершения. Также мы добавили признаки о динамике цены: была ли скидка и процент её падения.

In [10]:
# признаки этажности
df['is_first_floor'] = (df['floor'] == 1).astype('int8')
df['is_last_floor'] = (df['floor'] == df['total_floors']).astype('int8')
df['floor_ratio'] = (df['floor'] / df['total_floors']).replace([np.inf, -np.inf], np.nan)

# эффективность планировки
df['living_to_total'] = df['living_area'] / df['total_area']
df['kitchen_to_total'] = df['kitchen_area'] / df['total_area']
df['area_per_room'] = df['total_area'] / df['rooms'].replace(0, 1)  # студии считаем за 1 комнату, иначе делим на 0

# относительная цена по локации с учётом комнатности
for geo in ['district', 'municipality']:
    med = df.groupby([geo, 'rooms'])['price_per_m2'].transform('median')
    df[f'ppm2_to_{geo}'] = df['price_per_m2'] / med

df['nearest_metro_time'] = df['nearest_metro_time'].fillna(df['nearest_metro_time'].median())
df['total_lifts'] = df['passenger_lifts'].fillna(0) + df['cargo_lifts'].fillna(0)
df['has_lift'] = (df['total_lifts'] > 0).astype('int8')

Добавили признаки этажности (первый/последний этаж, отношение этажа к общему числу), эффективности планировки (доли жилой и кухонной площади, площадь на комнату) и относительной цены к медиане по району/муниципалитету и заполнили пропуски времени до метро.

In [11]:
def count_token(value, token):
    if pd.isna(value):
        return -1
    m = re.search(r'(\d+)\s*' + token, value)
    return int(m.group(1)) if m else 0

df['bath_separate'] = df['bathrooms'].map(lambda v: count_token(v, 'разд'))
df['bath_combined'] = df['bathrooms'].map(lambda v: count_token(v, 'совм'))
df['balcony_count'] = df['balcony'].map(lambda v: count_token(v, 'балк'))
df['loggia_count'] = df['balcony'].map(lambda v: count_token(v, 'лодж'))

### Удаляем "лишние" столбцы

In [12]:
drop_cols = [
    'last_seen', 'publication_date',
    'completion_date',
    'developer', 'residential_complex',
    'year_built', 'price',
    'price_min', 'price_max',
    'passenger_lifts', 'cargo_lifts',
    'bathrooms', 'balcony',
    'photos_count', 'views_today',
]
df = df.drop(columns=drop_cols)
df.columns

Index(['first_seen', 'days_on_market', 'event_closed', 'mortgage_allowed',
       'deal_conditions', 'region', 'municipality', 'district', 'lat', 'lon',
       'n_metro', 'nearest_metro', 'nearest_metro_time', 'nearest_metro_walk',
       'rooms', 'is_studio', 'flat_type', 'total_area', 'living_area',
       'kitchen_area', 'floor', 'total_floors', 'ceiling_height', 'renovation',
       'window_view', 'is_apartments', 'building_type', 'parking',
       'is_new_building', 'seller_type', 'phone_protected', 'views_total',
       'price_first', 'price_last', 'price_points', 'dist_to_center',
       'price_per_m2', 'building_age', 'is_ready', 'completion_year',
       'years_to_completion', 'is_presale', 'has_completion', 'had_discount',
       'price_drop_pct', 'is_first_floor', 'is_last_floor', 'floor_ratio',
       'living_to_total', 'kitchen_to_total', 'area_per_room',
       'ppm2_to_district', 'ppm2_to_municipality', 'total_lifts', 'has_lift',
       'bath_separate', 'bath_combined', 

Сформируем итоговый датасет

In [13]:
df.to_csv('./data/data_final.csv', index=False)
df

,first_seen,days_on_market,event_closed,mortgage_allowed,deal_conditions,region,municipality,district,lat,lon,n_metro,nearest_metro,nearest_metro_time,nearest_metro_walk,rooms,is_studio,flat_type,total_area,living_area,kitchen_area,floor,total_floors,ceiling_height,renovation,window_view,is_apartments,building_type,parking,is_new_building,seller_type,phone_protected,views_total,price_first,price_last,price_points,dist_to_center,price_per_m2,building_age,is_ready,completion_year,years_to_completion,is_presale,has_completion,had_discount,price_drop_pct,is_first_floor,is_last_floor,floor_ratio,living_to_total,kitchen_to_total,area_per_room,ppm2_to_district,ppm2_to_municipality,total_lifts,has_lift,bath_separate,bath_combined,balcony_count,loggia_count
0,2026-04-07 00:17:49.545645+00:00,3801,1,0,alternative,Москва,САО,Беговой,55.781467,37.561070,3,Беговая,13.0,0,3.0,0,unknown,73.8,45.3,8.8,3,14,2.80,cosmetic,yardAndStreet,0,brick,unknown,0,specialist,0,NaN,26000000,26000000,1,4.816265,352303.523035,56,True,NaN,NaN,0,0,0,0.000000,0,0,0.214286,0.613821,0.119241,24.600000,0.660886,0.898802,0.0,0,1,0,0,2
1,2026-04-06 21:40:26.848342+00:00,3766,1,0,alternative,Московская область,Реутов,unknown,55.763790,37.858013,3,Новогиреево,13.0,0,3.0,0,unknown,90.1,64.0,9.1,4,4,2.65,cosmetic,unknown,0,unknown,unknown,0,agency,0,NaN,16490000,16490000,1,15.105500,183018.867925,64,True,NaN,NaN,0,0,0,0.000000,0,1,1.000000,0.710322,0.100999,30.033333,0.855915,0.762837,0.0,0,1,0,2,0
2,2026-04-07 00:14:06.924608+00:00,3520,1,1,free,Москва,Красная Пахра село,unknown,55.431595,37.271490,3,Потапово,24.0,0,3.0,0,rooms,92.0,52.0,15.0,4,6,3.00,euro,yardAndStreet,0,brick,ground,0,specialist,1,167.0,22000000,22000000,1,41.737163,239130.434783,19,True,NaN,NaN,0,0,0,0.000000,0,0,0.666667,0.565217,0.163043,30.666667,1.118329,0.993272,0.0,0,1,1,0,1
3,2026-04-10 15:36:16.683205+00:00,3554,1,1,free,Московская область,Красногорск,unknown,55.813618,37.311794,3,Красногорская,8.0,1,2.0,0,rooms,48.0,36.0,7.5,2,5,2.80,euro,unknown,0,panel,ground,0,specialist,1,1476.0,10500000,10500000,1,20.306127,218750.000000,56,True,NaN,NaN,0,0,0,0.000000,0,0,0.400000,0.750000,0.156250,24.000000,0.970067,0.810226,0.0,0,0,1,1,0
4,2026-04-06 23:59:38.314351+00:00,3434,1,1,free,Московская область,Истра муниципальный округ,unknown,55.863490,37.045868,1,Нахабино,20.0,0,1.0,0,rooms,43.9,12.0,13.0,9,15,2.80,cosmetic,unknown,0,monolith,ground,0,specialist,1,1623.0,6700000,6950000,3,37.810351,152619.589977,13,True,NaN,NaN,0,0,0,-0.037313,0,0,0.600000,0.273349,0.296128,43.900000,0.615714,0.760341,2.0,1,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
142280,2026-06-16 17:38:34.834391+00:00,1,1,1,free,Москва,ЗАО,Проспект Вернадского,55.667675,37.510430,4,Новаторская,10.0,1,2.0,0,rooms,94.0,48.4,18.0,3,11,3.00,design,yard,0,monolith,underground,0,specialist,0,3.0,55000000,55000000,1,11.528608,585106.382979,19,True,2007.0,-19.0,0,1,0,0.000000,0,0,0.272727,0.514894,0.191489,47.000000,0.802837,0.985383,0.0,0,0,1,0,1
142281,2026-06-16 17:38:29.155986+00:00,0,1,1,alternative,Москва,unknown,Хорошёво-Мнёвники,55.770140,37.473680,4,Народное Ополчение,14.0,1,2.0,0,rooms,51.2,39.0,10.0,11,25,2.70,euro,street,0,monolith,underground,0,agency,1,14.0,26000000,26000000,1,9.221167,507812.500000,11,True,NaN,NaN,0,0,0,0.000000,0,0,0.440000,0.761719,0.195312,25.600000,0.821628,1.227501,4.0,1,0,1,-1,-1
142282,2026-06-16 17:45:39.568872+00:00,0,1,1,free,Москва,САО,Хорошевский,55.783140,37.509453,4,Зорге,13.0,1,3.0,0,rooms,67.7,47.7,20.0,9,22,3.20,design,unknown,1,monolith,underground,0,agency,0,15.0,51000000,51000000,1,7.594010,753323.485968,1,True,2023.0,-3.0,0,1,0,0.000000,0,0,0.409091,0.704579,0.295421,22.566667,1.340486,1.921891,3.0,1,1,0,-1,-1
142283,2026-06-16 17:43:56.902691+00:00,1,1,1,free,Москва,ЮЗАО,Коньково,55.6532